In [ ]:
#Importacion de librerias necesarias para el proyecto
import pandas as pd
import numpy as np
import requests
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

# Librerías para visualización
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## Carga y limpieza de datos

### Extracion de data, y eliminacion de columnas no deseadas

In [ ]:

#  Cargar JSON
df = pd.read_json('TelecomX_Data.json')
#df.head()

# Expandir cada columna anidada
customer_expanded = pd.json_normalize(df['customer'])
phone_expanded = pd.json_normalize(df['phone'])
internet_expanded = pd.json_normalize(df['internet'])
account_expanded = pd.json_normalize(df['account'])

# Combinar todo 
df_final = pd.concat([
    df[['customerID', 'Churn']],
    customer_expanded,
    phone_expanded,
    internet_expanded,
    account_expanded
], axis=1)

df_final.head()

In [ ]:
# Verificar si 'charges' está anidado y expandirlo
if 'charges' in df_final.columns:
    charges_expanded = pd.json_normalize(df_final['charges'])
    df_final = pd.concat([df_final.drop('charges', axis=1), charges_expanded], axis=1)
  
df = df_final  

### Tranfsformacion y Encoding de la data

In [ ]:
#  ELIMINAR CUSTOMERID (irrelevante)
df = df.drop('customerID', axis=1)


# LIMPIAR CHURN (eliminar filas con valor vacío)
filas_iniciales = len(df)
df = df[df['Churn'] != ''].copy()

#  LIMPIAR Y RELLENAR CHARGES.TOTAL
# Convertir a numérico (los espacios ' ' se vuelven NaN)
df['Charges.Total'] = pd.to_numeric(df['Charges.Total'], errors='coerce')
nan_count = df['Charges.Total'].isna().sum()
# Rellenar NaN con 0 (clientes nuevos)
df['Charges.Total'] = df['Charges.Total'].fillna(0)


#  TRANSFORMAR VARIABLES BINARIAS (Yes/No → 1/0)
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})


#  TRANSFORMAR GENDER (Female/Male → 1/0)
df['gender'] = df['gender'].map({'Female': 1, 'Male': 0})


#  TRANSFORMAR MULTIPLELINES (One-Hot Encoding)
multiple_dummies = pd.get_dummies(df['MultipleLines'], prefix='MultipleLines', drop_first=True)
df = pd.concat([df, multiple_dummies], axis=1)
df = df.drop('MultipleLines', axis=1)


# TRANSFORMAR INTERNETSERVICE (One-Hot Encoding)
internet_dummies = pd.get_dummies(df['InternetService'], prefix='InternetService', drop_first=True)
df = pd.concat([df, internet_dummies], axis=1)
df = df.drop('InternetService', axis=1)

# TRANSFORMAR SERVICIOS DE INTERNET
internet_services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                     'TechSupport', 'StreamingTV', 'StreamingMovies']

for service in internet_services:
    dummies = pd.get_dummies(df[service], prefix=service, drop_first=True)
    df = pd.concat([df, dummies], axis=1)
    df = df.drop(service, axis=1)


# TRANSFORMAR CONTRACT (One-Hot Encoding)
contract_dummies = pd.get_dummies(df['Contract'], prefix='Contract', drop_first=True)
df = pd.concat([df, contract_dummies], axis=1)
df = df.drop('Contract', axis=1)


# TRANSFORMAR PAYMENTMETHOD (One-Hot Encoding)
payment_dummies = pd.get_dummies(df['PaymentMethod'], prefix='PaymentMethod', drop_first=True)
df = pd.concat([df, payment_dummies], axis=1)
df = df.drop('PaymentMethod', axis=1)


# TRANSFORMAR CHURN (variable objetivo)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})


### Verificacion de la proporcion de cancelacion

In [ ]:
# Calcular la proporción de cancelación
churn_proportion = df['Churn'].value_counts(normalize=True) * 100

# Mostrar resultados de forma clara
print(" PROPORCIÓN DE CANCELACIÓN (CHURN):")
print("-" * 40)
print(churn_proportion)
print("-" * 40)

# También podemos ver los conteos absolutos
churn_counts = df['Churn'].value_counts()
print("\n CONTEO ABSOLUTO:")
print(f"Clientes activos (0): {churn_counts[0]}")
print(f"Clientes cancelados (1): {churn_counts[1]}")

In [ ]:

# 3. NORMALIZACIÓN DE DATOS (para modelos sensibles)

print("\n NORMALIZACIÓN DE DATOS:")
print("-" * 50)
print("""
¿Por qué normalizar?
- Modelos como Regresión Logística y KNN usan distancias/gradientes
- Variables con escalas grandes (Charges.Total) dominarían a las pequeñas (0/1)
- Estandarización: media=0, desviación=1 para todas las variables
""")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Datos normalizados con StandardScaler")
print(f"   Media (primeras 5 variables): {X_train_scaled.mean(axis=0)[:5].round(2)}")
print(f"   Std (primeras 5 variables): {X_train_scaled.std(axis=0)[:5].round(2)}")

### Correlacion

In [ ]:
# PREPARAR DATOS
# Asegurarnos que todas las columnas sean numéricas
df_numerico = df.select_dtypes(include=[np.number])

print(f"\n Dimensiones del dataset numérico: {df_numerico.shape}")
print(f"\n Columnas incluidas ({len(df_numerico.columns)}):")
for i, col in enumerate(df_numerico.columns[:10]):  # Mostrar primeras 10
    print(f"   {i+1}. {col}")
if len(df_numerico.columns) > 10:
    print(f"   ... y {len(df_numerico.columns) - 10} más")

In [ ]:
# CALCULAR MATRIZ DE CORRELACIÓN

correlation_matrix = df_numerico.corr()

#  VISUALIZAR MATRIZ COMPLETA (MAPA DE CALOR)

plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, 
            annot=False,  # Cambiar a True si quieres ver números
            cmap='RdBu_r', 
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8})

In [ ]:

# ENFOQUE EN CORRELACIÓN CON CHURN 

print("\n" + "=" * 70)
print(" CORRELACIÓN CON LA VARIABLE OBJETIVO (CHURN)")
print("=" * 70)

# Obtener correlaciones con Churn y ordenarlas
corr_con_churn = correlation_matrix['Churn'].drop('Churn').sort_values(ascending=False)

# Convertir a DataFrame para mejor visualización
df_corr_churn = pd.DataFrame({
    'Variable': corr_con_churn.index,
    'Correlación con Churn': corr_con_churn.values,
    '|Correlación|': np.abs(corr_con_churn.values)
}).sort_values('|Correlación|', ascending=False)

print("\n TOP 15 VARIABLES MÁS CORRELACIONADAS CON CHURN:")
print("-" * 70)
print(df_corr_churn.head(15).to_string(index=False))


In [ ]:
# VISUALIZACIÓN GRÁFICA DE CORRELACIONES CON CHURN

plt.figure(figsize=(12, 10))

# Seleccionar top 20 variables (o todas si hay menos)
top_n = min(20, len(df_corr_churn))
top_corr = df_corr_churn.head(top_n)

# Crear barras horizontales
colors = ['crimson' if x > 0 else 'steelblue' for x in top_corr['Correlación con Churn']]
plt.barh(range(top_n), top_corr['Correlación con Churn'], color=colors)
plt.yticks(range(top_n), top_corr['Variable'])
plt.xlabel('Correlación con Churn', fontsize=12)
plt.title(f'Top {top_n} Variables más Correlacionadas con Cancelación (Churn)', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:

# ANÁLISIS DE MULTICOLINEALIDAD (VARIABLES REDUNDANTES)

print("\n" + "=" * 70)
print("ANÁLISIS DE MULTICOLINEALIDAD (VARIABLES REDUNDANTES)")
print("=" * 70)

# Encontrar pares con alta correlación (> 0.7)
high_corr_pairs = []
threshold = 0.7

for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > threshold:
            high_corr_pairs.append({
                'Variable 1': correlation_matrix.columns[i],
                'Variable 2': correlation_matrix.columns[j],
                'Correlación': correlation_matrix.iloc[i, j]
            })

if high_corr_pairs:
    df_high_corr = pd.DataFrame(high_corr_pairs).sort_values('Correlación', ascending=False)
    print(f"\n PARES CON ALTA CORRELACIÓN (> {threshold}):")
    print("-" * 70)
    print(df_high_corr.to_string(index=False))
    
    # Análisis de posibles redundancias
    print("\n RECOMENDACIONES:")
    print("   - Considerar eliminar una variable de cada par redundante")
    print("   - Priorizar mantener la variable con mayor correlación con Churn")
    print("   - Técnicas como PCA pueden ayudar si hay mucha redundancia")
else:
    print(f"\n No se encontraron pares con correlación > {threshold}")



In [ ]:

#  ANÁLISIS ESPECÍFICO DE VARIABLES CLAVE

print("\n" + "=" * 70)
print("🔍 ANÁLISIS DETALLADO DE VARIABLES CLAVE")
print("=" * 70)

# Identificar variables con mayor correlación positiva y negativa
top_positivas = corr_con_churn.head(5)
top_negativas = corr_con_churn.tail(5)

print("\n TOP 5 VARIABLES CON CORRELACIÓN POSITIVA (AUMENTAN CHURN):")
for var, corr in top_positivas.items():
    print(f"   ✅ {var}: +{corr:.3f}")

print("\nTOP 5 VARIABLES CON CORRELACIÓN NEGATIVA (DISMINUYEN CHURN):")
for var, corr in top_negativas.items():
    print(f"   ❌ {var}: {corr:.3f}")

### Analisis Dirigido

In [ ]:

#  TIEMPO DE CONTRATO (TENURE) VS CANCELACIÓN
print("-" * 70)
print("RELACIÓN: TIEMPO DE CONTRATO × CANCELACIÓN")
print("-" * 70)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Gráfico 1: Boxplot de tenure por Churn
ax1 = axes[0, 0]
sns.boxplot(x='Churn', y='tenure', data=df, ax=ax1, palette=['#3498db', '#e74c3c'])
ax1.set_xlabel('Churn (0=No cancela, 1=Sí cancela)', fontsize=11)
ax1.set_ylabel('Tiempo de contrato (meses)', fontsize=11)
ax1.set_title('Distribución del Tiempo de Contrato por Cancelación', fontsize=12, fontweight='bold')

# Gráfico 2: Histograma superpuesto
ax2 = axes[0, 1]
sns.histplot(data=df, x='tenure', hue='Churn', multiple='stack', 
             palette=['#3498db', '#e74c3c'], alpha=0.7, ax=ax2, bins=30)
ax2.set_xlabel('Tiempo de contrato (meses)', fontsize=11)
ax2.set_ylabel('Cantidad de clientes', fontsize=11)
ax2.set_title('Distribución de Tiempo de Contrato (apilado)', fontsize=12, fontweight='bold')
ax2.legend(['No Cancela', 'Cancela'])

# Gráfico 3: Violin plot
ax3 = axes[1, 0]
sns.violinplot(x='Churn', y='tenure', data=df, ax=ax3, palette=['#3498db', '#e74c3c'])
ax3.set_xlabel('Churn (0=No cancela, 1=Sí cancela)', fontsize=11)
ax3.set_ylabel('Tiempo de contrato (meses)', fontsize=11)
ax3.set_title('Distribución de Densidad - Tiempo de Contrato', fontsize=12, fontweight='bold')

# Gráfico 4: Tasa de churn por rangos de tenure (CORREGIDO)
ax4 = axes[1, 1]

# Crear rangos de tenure
df['tenure_range'] = pd.cut(df['tenure'], bins=[0, 6, 12, 24, 48, 72], 
                             labels=['0-6 meses', '6-12 meses', '1-2 años', '2-4 años', '4+ años'])

# Calcular tasa de churn por rango (CORREGIDO: no multiplicamos el DataFrame completo)
churn_by_tenure = df.groupby('tenure_range')['Churn'].mean().reset_index()
churn_by_tenure['Churn'] = churn_by_tenure['Churn'] * 100  # Multiplicamos solo la columna numérica

sns.barplot(x='tenure_range', y='Churn', data=churn_by_tenure, ax=ax4, palette='Reds_r')
ax4.set_xlabel('Tiempo de contrato', fontsize=11)
ax4.set_ylabel('Tasa de cancelación (%)', fontsize=11)
ax4.set_title('Tasa de Cancelación por Rango de Tenure', fontsize=12, fontweight='bold')
ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()



In [ ]:

# GASTO TOTAL (CHARGES.TOTAL) VS CANCELACIÓN

print("\n" + "=" * 70)
print(" RELACIÓN: GASTO TOTAL × CANCELACIÓN")
print("-" * 70)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Gráfico 1: Boxplot de Charges.Total por Churn
ax1 = axes[0, 0]
sns.boxplot(x='Churn', y='Charges.Total', data=df, ax=ax1, palette=['#3498db', '#e74c3c'])
ax1.set_xlabel('Churn (0=No cancela, 1=Sí cancela)', fontsize=11)
ax1.set_ylabel('Gasto Total ($)', fontsize=11)
ax1.set_title('Distribución del Gasto Total por Cancelación', fontsize=12, fontweight='bold')

# Gráfico 2: Histograma superpuesto
ax2 = axes[0, 1]
sns.histplot(data=df, x='Charges.Total', hue='Churn', multiple='stack',
             palette=['#3498db', '#e74c3c'], alpha=0.7, ax=ax2, bins=30)
ax2.set_xlabel('Gasto Total ($)', fontsize=11)
ax2.set_ylabel('Cantidad de clientes', fontsize=11)
ax2.set_title('Distribución de Gasto Total (apilado)', fontsize=12, fontweight='bold')
ax2.legend(['No Cancela', 'Cancela'])

# Gráfico 3: Scatter plot tenure vs charges.total coloreado por churn
ax3 = axes[1, 0]
scatter = ax3.scatter(df['tenure'], df['Charges.Total'], 
                      c=df['Churn'], cmap='coolwarm', alpha=0.6, s=20)
ax3.set_xlabel('Tiempo de contrato (meses)', fontsize=11)
ax3.set_ylabel('Gasto Total ($)', fontsize=11)
ax3.set_title('Relación: Tenure vs Gasto Total (color = Churn)', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax3, label='Churn (0=No, 1=Sí)')

# Gráfico 4: Tasa de churn por rangos de gasto (CORREGIDO)
ax4 = axes[1, 1]

# Crear rangos de gasto total (usando qcut para quintiles)
# Primero filtramos valores cero para los quintiles
df_temp = df[df['Charges.Total'] > 0].copy()
df_temp['total_charges_range'] = pd.qcut(df_temp['Charges.Total'], 
                                         q=5, labels=['Muy bajo', 'Bajo', 'Medio', 'Alto', 'Muy alto'])

# Merge de vuelta al DataFrame original
df['total_charges_range'] = None
df.loc[df['Charges.Total'] > 0, 'total_charges_range'] = df_temp['total_charges_range']
df.loc[df['Charges.Total'] == 0, 'total_charges_range'] = 'Cero'

# Calcular tasa de churn por rango de gasto (CORREGIDO)
churn_by_charges = df.groupby('total_charges_range')['Churn'].mean().reset_index()
churn_by_charges['Churn'] = churn_by_charges['Churn'] * 100  # Multiplicamos solo la columna numérica

sns.barplot(x='total_charges_range', y='Churn', data=churn_by_charges, ax=ax4, palette='viridis')
ax4.set_xlabel('Nivel de Gasto Total', fontsize=11)
ax4.set_ylabel('Tasa de cancelación (%)', fontsize=11)
ax4.set_title('Tasa de Cancelación por Nivel de Gasto', fontsize=12, fontweight='bold')
ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()



In [ ]:

#  ANÁLISIS COMBINADO: TENURE × GASTO × CHURN

print("\n" + "=" * 70)
print("  ANÁLISIS COMBINADO: TENURE × GASTO TOTAL × CHURN")
print("-" * 70)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Gráfico 1: Relación tenure-gasto con líneas de tendencia
ax1 = axes[0]
for churn_value, color, label in [(0, '#3498db', 'No Cancela'), (1, '#e74c3c', 'Cancela')]:
    subset = df[df['Churn'] == churn_value]
    ax1.scatter(subset['tenure'], subset['Charges.Total'], 
                c=color, label=label, alpha=0.4, s=15)
    # Línea de tendencia (solo si hay suficientes puntos)
    if len(subset) > 5:
        z = np.polyfit(subset['tenure'], subset['Charges.Total'], 1)
        p = np.poly1d(z)
        ax1.plot(sorted(subset['tenure']), p(sorted(subset['tenure'])), 
                 color=color, linewidth=2, linestyle='--')

ax1.set_xlabel('Tiempo de contrato (meses)', fontsize=11)
ax1.set_ylabel('Gasto Total ($)', fontsize=11)
ax1.set_title('Relación Tenure-Gasto Total con Líneas de Tendencia', fontsize=12, fontweight='bold')
ax1.legend()

# Gráfico 2: Tasa de churn por cuadrantes (CORREGIDO)
ax2 = axes[1]

# Crear puntos de corte (medianas)
tenure_median = df['tenure'].median()
charges_median = df['Charges.Total'].median()

# Crear cuadrantes
df['cuadrante'] = 'Bajo tenure, Bajo gasto'
df.loc[(df['tenure'] >= tenure_median) & (df['Charges.Total'] < charges_median), 'cuadrante'] = 'Alto tenure, Bajo gasto'
df.loc[(df['tenure'] < tenure_median) & (df['Charges.Total'] >= charges_median), 'cuadrante'] = 'Bajo tenure, Alto gasto'
df.loc[(df['tenure'] >= tenure_median) & (df['Charges.Total'] >= charges_median), 'cuadrante'] = 'Alto tenure, Alto gasto'

# Calcular tasa de churn por cuadrante (CORREGIDO)
churn_by_cuadrante = df.groupby('cuadrante')['Churn'].mean().reset_index()
churn_by_cuadrante['Churn'] = churn_by_cuadrante['Churn'] * 100  # Multiplicamos solo la columna numérica

sns.barplot(x='cuadrante', y='Churn', data=churn_by_cuadrante, ax=ax2, palette='coolwarm')
ax2.set_xlabel('Segmento de clientes', fontsize=11)
ax2.set_ylabel('Tasa de cancelación (%)', fontsize=11)
ax2.set_title('Tasa de Cancelación por Segmento Tenure-Gasto', fontsize=12, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()



In [ ]:

# 4. ESTADÍSTICAS DESCRIPTIVAS

print("\n" + "=" * 70)
print(" ESTADÍSTICAS DESCRIPTIVAS")
print("=" * 70)

print("\n ESTADÍSTICAS DE TENURE POR CHURN:")
print(df.groupby('Churn')['tenure'].describe())

print("\n ESTADÍSTICAS DE GASTO TOTAL POR CHURN:")
print(df.groupby('Churn')['Charges.Total'].describe())

print("\n TASA DE CANCELACIÓN POR SEGMENTO:")
segment_stats = df.groupby('cuadrante')['Churn'].agg(['count', 'mean']).round(3)
segment_stats['mean'] = segment_stats['mean'] * 100
print(segment_stats)


In [ ]:

# INTERPRETACIÓN ESTRATÉGICA

print("\n" + "=" * 70)
print(" INTERPRETACIÓN ESTRATÉGICA PARA TELECOM X")
print("=" * 70)

# Calcular métricas clave
churn_rate_early = df[df['tenure'] < 12]['Churn'].mean() * 100
churn_rate_late = df[df['tenure'] >= 12]['Churn'].mean() * 100
avg_tenure_churn = df[df['Churn'] == 1]['tenure'].mean()
avg_tenure_no_churn = df[df['Churn'] == 0]['tenure'].mean()
avg_charges_churn = df[df['Churn'] == 1]['Charges.Total'].mean()
avg_charges_no_churn = df[df['Churn'] == 0]['Charges.Total'].mean()

print(f"""
 CONCLUSIONES ESTRATÉGICAS:

1.  TIEMPO DE CONTRATO:
    - Clientes que cancelan tienen en promedio {avg_tenure_churn:.1f} meses de antigüedad
    - Clientes que permanecen tienen en promedio {avg_tenure_no_churn:.1f} meses
    - Tasa de cancelación en primeros 12 meses: {churn_rate_early:.1f}%
    - Tasa de cancelación después de 12 meses: {churn_rate_late:.1f}%

     ACCIÓN: Implementar programa de retención en primeros 6 meses

2.  GASTO TOTAL:
    - Clientes que cancelan: gasto promedio ${avg_charges_churn:.0f}
    - Clientes que permanecen: gasto promedio ${avg_charges_no_churn:.0f}
    
     ACCIÓN: Los clientes de bajo gasto (nuevos) son los más riesgosos

3.  SEGMENTOS DE RIESGO:
    - Mayor riesgo: Clientes con bajo tenure y bajo gasto (nuevos)
    - Menor riesgo: Clientes con alto tenure y alto gasto (leales)
    
     ACCIÓN: Enfocar esfuerzos en retener clientes nuevos
""")

## Modelado Predictivo

### Separacion de datos

### Normalizacion o estandarizacion para modelos sensibles

In [ ]:

# NORMALIZACIÓN DE DATOS (para modelos sensibles)

print("\n NORMALIZACIÓN DE DATOS:")
print("-" * 50)
print("""
¿Por qué normalizar?
- Modelos como Regresión Logística y KNN usan distancias/gradientes
- Variables con escalas grandes (Charges.Total) dominarían a las pequeñas (0/1)
- Estandarización: media=0, desviación=1 para todas las variables
""")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Datos normalizados con StandardScaler")
print(f"   Media (primeras 5 variables): {X_train_scaled.mean(axis=0)[:5].round(2)}")
print(f"   Std (primeras 5 variables): {X_train_scaled.std(axis=0)[:5].round(2)}")

### Creacion de modelos predictivos

In [ ]:

# DEFINICIÓN DE MODELOS

print("\n" + "=" * 70)
print("🎯 MODELOS SELECCIONADOS")
print("=" * 70)

modelos = {
    # MODELOS SENSIBLES A ESCALA (usan datos normalizados)
    'Sensibles a escala': {
        'Regresión Logística': LogisticRegression(
            class_weight='balanced',  # Maneja desbalance
            max_iter=1000,
            random_state=42
        ),
        'KNN': KNeighborsClassifier(
            n_neighbors=5,
            weights='distance'
        )
    },
    
    # MODELOS NO SENSIBLES A ESCALA (usan datos originales)
    'No sensibles a escala': {
        'Árbol de Decisión': DecisionTreeClassifier(
            class_weight='balanced',
            max_depth=5,  # Limitar profundidad para evitar overfitting
            random_state=42
        ),
        'Random Forest': RandomForestClassifier(
            class_weight='balanced',
            n_estimators=100,
            max_depth=10,
            random_state=42
        ),
        'XGBoost': XGBClassifier(
            scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),  # Balanceo
            n_estimators=100,
            max_depth=6,
            random_state=42
        )
    }
}

print("\n📋 Modelos a entrenar:")

print("\n🔵 MODELOS SENSIBLES A ESCALA (con normalización):")
for nombre in modelos['Sensibles a escala'].keys():
    print(f"   - {nombre}")

print("\n🟢 MODELOS NO SENSIBLES A ESCALA (sin normalización):")
for nombre in modelos['No sensibles a escala'].keys():
    print(f"   - {nombre}")

# ============================================================
# 5. ENTRENAMIENTO Y EVALUACIÓN
# ============================================================
print("\n" + "=" * 70)
print("📊 ENTRENAMIENTO Y EVALUACIÓN")
print("=" * 70)

resultados = []

# Función para evaluar modelo
def evaluar_modelo(modelo, X_train_data, X_test_data, y_train, y_test, nombre, tipo):
    # Entrenar
    modelo.fit(X_train_data, y_train)
    
    # Predecir
    y_pred = modelo.predict(X_test_data)
    y_proba = modelo.predict_proba(X_test_data)[:, 1]
    
    # Métricas
    metrics = {
        'Modelo': nombre,
        'Tipo': tipo,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba)
    }
    
    return metrics, y_pred, y_proba